# Evaluate the dev set

Run the cells in order to generate predictions for all 60 dev notes, starting with `dev-001`, and evaluate them using `score.py`. The prediction step sends note text to the configured model provider and incurs API charges. It overwrites the prediction and per-note results files in `out/`.

Set `OPENAI_API_KEY` in your environment or project `.env` before running. Edit the provider or model below if needed.

In [1]:
import html
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import HTML, display

root = Path.cwd()
if not (root / 'llm.py').is_file() or not (root / 'score.py').is_file():
    raise FileNotFoundError('Start the notebook with the project root as its working directory.')

provider = 'openai'  # or 'openrouter'
model = None  # None uses llm.py's provider default or model environment variable
gold_path = root / 'data' / 'dev_notes.jsonl'
mms_path = root / 'data' / 'mms-2026-01.json'
output_dir = root / 'out'
predictions_path = output_dir / 'dev_predictions.jsonl'
per_note_path = output_dir / 'dev_per_note.jsonl'

## Generate predictions

This makes one model request per note. The script writes each prediction as it completes. If a request fails, fix the error before rerunning; rerunning starts from the beginning and overwrites the previous predictions.

In [3]:
output_dir.mkdir(parents=True, exist_ok=True)
command = ['/opt/homebrew/bin/python3', str(root / 'llm.py'), str(gold_path), str(predictions_path), '--provider', provider]
if model is not None:
    command.extend(['--model', model])
print('Generating predictions; this can take several minutes.', flush=True)
subprocess.run(command, cwd=root, check=True)
print(f'Predictions saved to {predictions_path}')

Generating predictions; this can take several minutes.
Predictions saved to /Users/shivangi.s.1/Desktop/sugen_test/out/dev_predictions.jsonl


## Score predictions

The scorer reports top-1 and top-3 accuracy by slice and the official weighted score. Missing predictions count as misses, including if you score an interrupted run.

In [4]:
subprocess.run([
    sys.executable, str(root / 'score.py'),
    '--gold', str(gold_path),
    '--pred', str(predictions_path),
    '--mms', str(mms_path),
    '--per-note', str(per_note_path),
], cwd=root, check=True);

notes 60  missing predictions 0  invalid codes 0  duplicate ids 0
slice A: n=60  top1 0.217  top3 0.217  points 16.2 / 75
SCORE 16.2 / 75


## Per-note results

Compare the gold and predicted codes for every note. `top1` and `top3` follow the scorer's normalization and validity rules.

In [5]:
with per_note_path.open(encoding='utf-8') as file:
    results = [json.loads(line) for line in file if line.strip()]

columns = ('id', 'slice', 'gold', 'pred', 'top1', 'top3')
header = ''.join(f'<th>{html.escape(column)}</th>' for column in columns)
rows = []
for result in results:
    values = (result[column] for column in columns)
    cells = ''.join(f'<td>{html.escape(str(value))}</td>' for value in values)
    rows.append(f'<tr>{cells}</tr>')
print(f'{len(results)} notes evaluated')
display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{"".join(rows)}</tbody></table>'))

60 notes evaluated


id,slice,gold,pred,top1,top3
dev-001,A,['1F40.Z'],['1F40'],False,False
dev-002,A,"['CA00', 'CA07.0']",['CA00'],True,True
dev-003,A,['CA23.30'],[],False,False
dev-004,A,['1A07.Z'],['1A07.0'],False,False
dev-005,A,['1F23.10'],['1F23.3'],False,False
dev-006,A,['1E90.0'],['1E90.0'],True,True
dev-007,A,['CA40.Z'],['CA40.Z'],True,True
dev-008,A,['1G05'],['1G03'],False,False
dev-009,A,['1B10.0'],['1B10'],False,False
dev-010,A,['CA03.Z'],['CA02.Z'],False,False
